# Main runs, alternative reading of the bounds

**Computes:** FCM, the type-1 control and the interval algorithm (alternative reading) on seventeen data sets; partition against nearest prototype; the best pair per instance against FCM at m = 2 (the design of Hwang and Rhee).

**Inputs:** UCI data (downloaded), DIBCO images.  
**Outputs:** `results/05/results_05.json` and a text report.


In [ ]:
import os, sys, time, json, pickle, platform, subprocess, glob, urllib.request
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DATASETS = ("equal", "volume", "noise", "density", "volume3",
            "iris", "wine", "wdbc", "wbc", "sonar", "pima", "seeds", "glass", "ionosphere", "banknote", "ecoli", "dibco")
RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")

RESULTS_DIR = os.environ.get("RESULTS_DIR", "results")
OUTDIR = os.path.join(RESULTS_DIR, "05")
OUTDIR = os.path.abspath(OUTDIR)
for sub in ("data", "reports"):
    os.makedirs(os.path.join(OUTDIR, sub), exist_ok=True)
import numpy as np, scipy, sklearn
print(f"colab {IN_COLAB} | python {platform.python_version()} | numpy {np.__version__} | scipy {scipy.__version__} | sklearn {sklearn.__version__}")
print(f"output dir: {OUTDIR}   run stamp: {RUN_STAMP}")

In [ ]:
MIRROR = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/"
FILES = ["sonar.csv", "pima-indians-diabetes.csv", "breast-cancer-wisconsin.data", "wheat-seeds.csv",
         "glass.csv", "ionosphere.csv", "banknote_authentication.csv", "ecoli.csv"]
DATA_DIR = os.path.join(OUTDIR, "data")
for f in FILES:
    p = os.path.join(DATA_DIR, f)
    if not os.path.exists(p):
        urllib.request.urlretrieve(MIRROR + f, p)
print("tabular files:", sorted(os.listdir(DATA_DIR)))
DIBCO_ROOT = os.environ.get("DIBCO_ROOT") or ("/content/dibco_all" if IN_COLAB else os.path.join(OUTDIR, "dibco_all"))
DIBCO_DIR = os.path.join(DIBCO_ROOT, "repo", "SauvolaNet", "Dataset")
if "dibco" in DATASETS and not os.path.isdir(DIBCO_DIR):
    print("cloning DIBCO (~945 MB) to", DIBCO_ROOT, "...")
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/RichSu95/Document_Binarization_Collection.git",
                    os.path.join(DIBCO_ROOT, "repo")], check=True)
if "dibco" in DATASETS:
    print(len(glob.glob(os.path.join(DIBCO_DIR, "*DIBCO*_source.*"))), "DIBCO images")

In [ ]:
import numpy as np
from itertools import permutations

EPS = 1e-12

def fcm_membership(D, m):
    p = 2.0 / (m - 1.0)
    z = -p * np.log(np.maximum(D, EPS))
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def hr_bounds(D, m1, m2):
    C = D.shape[1]
    U1, U2 = fcm_membership(D, m1), fcm_membership(D, m2)
    s = 1.0 / (D[:, :, None] / np.maximum(D[:, None, :], EPS)).sum(axis=2)
    cond = s < 1.0 / C
    upper = np.where(cond, U1, U2)
    lower = np.where(cond, U2, U1)
    return lower, upper

def km_interval_1d(x, wl, wu):
    o = np.argsort(x, kind="stable"); xs = x[o]; l = wl[o]; u = wu[o]
    n = len(x)
    cl, cu = np.cumsum(l), np.cumsum(u)
    clx, cux = np.cumsum(l * xs), np.cumsum(u * xs)
    Tl, Tu, Tlx, Tux = cl[-1], cu[-1], clx[-1], cux[-1]
    k = np.arange(n + 1)
    Lk = np.concatenate([[0.0], cl]); Uk = np.concatenate([[0.0], cu])
    Lxk = np.concatenate([[0.0], clx]); Uxk = np.concatenate([[0.0], cux])
    vR_all = (Lxk + (Tux - Uxk)) / np.maximum(Lk + (Tu - Uk), EPS)
    vL_all = (Uxk + (Tlx - Lxk)) / np.maximum(Uk + (Tl - Lk), EPS)
    kR, kL = int(np.argmax(vR_all)), int(np.argmin(vL_all))
    selR = np.zeros(n, bool); selL = np.zeros(n, bool)
    rank = np.empty(n, int); rank[o] = np.arange(n)
    selR[:] = rank >= kR
    selL[:] = rank < kL
    return float(vL_all[kL]), float(vR_all[kR]), selL, selR

def it2fcm_update(X, lower, upper, mc):
    N, M = X.shape; C = lower.shape[1]
    V = np.zeros((C, M)); uL = np.zeros((N, C)); uR = np.zeros((N, C))
    wl, wu = lower ** mc, upper ** mc
    for j in range(C):
        accL = np.zeros(N); accR = np.zeros(N)
        for l in range(M):
            vL, vR, selL, selR = km_interval_1d(X[:, l], wl[:, j], wu[:, j])
            V[j, l] = 0.5 * (vL + vR)
            accL += np.where(selL, upper[:, j], lower[:, j])
            accR += np.where(selR, upper[:, j], lower[:, j])
        uL[:, j], uR[:, j] = accL / M, accR / M
    return V, uL, uR

def dist(X, V):
    return np.maximum(np.linalg.norm(X[:, None, :] - V[None, :, :], axis=2), EPS)

def run_fcm(X, V0, m, iters=300, tol=1e-7):
    V = V0.copy()
    for _ in range(iters):
        U = fcm_membership(dist(X, V), m); W = U ** m
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    U = fcm_membership(dist(X, V), m)
    return V, U.argmax(1)

def run_it2fcm(X, V0, m1, m2, mc=None, iters=300, tol=1e-7):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update(X, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update(X, lo, up, mc)
    hard_hr = (0.5 * (uL + uR)).argmax(1)
    hard_nearest = dist(X, V).argmin(1)
    return V, hard_hr, hard_nearest

def accuracy(labels, pred, C):
    best = 0.0
    for perm in permutations(range(C)):
        best = max(best, float(np.mean(np.array(perm)[pred] == labels)))
    return best

def centre_error(V, centres):
    C = len(centres); best = np.inf
    for perm in permutations(range(C)):
        best = min(best, float(np.mean(np.linalg.norm(V[list(perm)] - centres, axis=1))))
    return best

def init_pp(X, C, rng):
    idx = [rng.integers(len(X))]
    for _ in range(1, C):
        d2 = np.min(((X[:, None, :] - X[idx][None]) ** 2).sum(2), axis=1)
        idx.append(rng.choice(len(X), p=d2 / d2.sum()))
    return X[idx].copy()

def scenario(name, seed):
    r = np.random.default_rng(seed)
    if name == "equal":
        c = np.array([[0.0, 0.0], [4.0, 0.0]]); n = (100, 100); s = (1.0, 1.0)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(2)])
    elif name == "volume":
        c = np.array([[0.0, 0.0], [5.0, 0.0]]); n = (100, 100); half = (2.5, 0.6)
        X = np.vstack([c[k] + r.uniform(-half[k], half[k], size=(n[k], 2)) for k in range(2)])
    elif name == "noise":
        c = np.array([[0.0, 0.0], [0.0, 4.0]]); n = (100, 100)
        X = np.vstack([r.normal(c[0], 1.0, size=(n[0], 2)), r.normal(c[1], 1.0, size=(n[1], 2)),
                       r.normal(c[0], 3.0, size=(40, 2))])
        y = np.r_[np.zeros(n[0]), np.ones(n[1]), np.zeros(40)].astype(int)
        return X, y, c, 2
    elif name == "density":
        c = np.array([[0.0, 0.0], [3.5, 0.0]]); n = (200, 30); s = (1.0, 1.0)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(2)])
    elif name == "volume3":
        c = np.array([[0.0, 0.0], [5.0, 0.0], [2.5, 4.0]]); n = (100, 100, 100); s = (1.4, 0.4, 0.8)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(3)])
    else:
        raise ValueError(name)
    y = np.concatenate([np.full(n[k], k) for k in range(len(n))])
    return X, y, c, len(n)

from scipy.optimize import linear_sum_assignment
def accuracy_h(labels, pred, C):
    M = np.zeros((C, C))
    np.add.at(M, (labels, pred), 1)
    r, c = linear_sum_assignment(-M)
    return float(M[r, c].sum() / len(labels))

def align(V, ref):
    Dm = np.linalg.norm(V[:, None] - ref[None], axis=2)
    r, c = linear_sum_assignment(Dm)
    out = np.zeros_like(ref); out[c] = V[r]
    return out

def centre_error_h(V, centres):
    return float(np.mean(np.linalg.norm(align(V, centres) - centres, axis=1)))

def real_dataset(name):
    from sklearn import datasets
    d = {"iris": datasets.load_iris, "wine": datasets.load_wine, "breast_cancer": datasets.load_breast_cancer}[name]()
    X = d.data.astype(float); X = (X - X.mean(0)) / X.std(0)
    y = d.target.astype(int)
    return X, y, len(np.unique(y))

def run_fcm_decoupled(X, V0, ma, mb, iters=300, tol=1e-7):
    V = V0.copy()
    for _ in range(iters):
        U = fcm_membership(dist(X, V), ma); W = U ** mb
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    return V, dist(X, V).argmin(1)

def it2fcm_update_fast(X, order, lower, upper, mc):
    N, M = X.shape; C = lower.shape[1]
    Xs = np.take_along_axis(X, order, axis=0)
    rank = np.empty_like(order); np.put_along_axis(rank, order, np.arange(N)[:, None].repeat(M, 1), axis=0)
    V = np.zeros((C, M)); uL = np.zeros((N, C)); uR = np.zeros((N, C))
    z = np.zeros((1, M))
    for j in range(C):
        wl = (lower[:, j] ** mc)[order]; wu = (upper[:, j] ** mc)[order]
        cl, cu = np.cumsum(wl, 0), np.cumsum(wu, 0)
        clx, cux = np.cumsum(wl * Xs, 0), np.cumsum(wu * Xs, 0)
        Lk, Uk = np.vstack([z, cl]), np.vstack([z, cu]); Lxk, Uxk = np.vstack([z, clx]), np.vstack([z, cux])
        Tl, Tu, Tlx, Tux = cl[-1], cu[-1], clx[-1], cux[-1]
        vR_all = (Lxk + (Tux - Uxk)) / np.maximum(Lk + (Tu - Uk), EPS)
        vL_all = (Uxk + (Tlx - Lxk)) / np.maximum(Uk + (Tl - Lk), EPS)
        kR, kL = vR_all.argmax(0), vL_all.argmin(0)
        vR = vR_all[kR, np.arange(M)]; vL = vL_all[kL, np.arange(M)]
        V[j] = 0.5 * (vL + vR)
        selR = rank >= kR[None, :]; selL = rank < kL[None, :]
        up, lo = upper[:, j][:, None], lower[:, j][:, None]
        uR[:, j] = np.where(selR, up, lo).mean(1); uL[:, j] = np.where(selL, up, lo).mean(1)
    return V, uL, uR

def run_it2fcm_fast(X, V0, m1, m2, mc=None, iters=300, tol=1e-7, order=None):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    order = np.argsort(X, axis=0, kind="stable") if order is None else order
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
    return V, (0.5 * (uL + uR)).argmax(1), dist(X, V).argmin(1)

def real_dataset2(name, data_dir=DATA_DIR):
    import os, pandas as pd
    if name in ("iris", "wine", "wdbc"):
        return real_dataset({"wdbc": "breast_cancer"}.get(name, name))
    if name == "sonar":
        d = pd.read_csv(os.path.join(data_dir, "sonar.csv"), header=None)
        X = d.iloc[:, :-1].to_numpy(float); y = (d.iloc[:, -1] == "M").to_numpy(int)
    elif name == "pima":
        d = pd.read_csv(os.path.join(data_dir, "pima-indians-diabetes.csv"), header=None)
        X = d.iloc[:, :-1].to_numpy(float); y = d.iloc[:, -1].to_numpy(int)
    elif name == "wbc":
        d = pd.read_csv(os.path.join(data_dir, "breast-cancer-wisconsin.data"), header=None, na_values="?").dropna()
        X = d.iloc[:, 1:-1].to_numpy(float); y = (d.iloc[:, -1] == 4).to_numpy(int)
    else:
        raise ValueError(name)
    X = (X - X.mean(0)) / X.std(0)
    return X, y, len(np.unique(y))

EXTRA = {"seeds": "wheat-seeds.csv", "glass": "glass.csv", "ionosphere": "ionosphere.csv",
         "banknote": "banknote_authentication.csv", "ecoli": "ecoli.csv"}
def real_dataset3(name, data_dir=DATA_DIR):
    import os, pandas as pd
    if name not in EXTRA:
        return real_dataset2(name, data_dir)
    d = pd.read_csv(os.path.join(data_dir, EXTRA[name]), header=None)
    X = d.iloc[:, :-1].to_numpy(float)
    y = pd.factorize(d.iloc[:, -1].astype(str).str.strip(), sort=True)[0]
    X = X[:, X.std(0) > 0]
    X = (X - X.mean(0)) / X.std(0)
    return X, y.astype(int), len(np.unique(y))

In [ ]:
GRID = (1.05, 1.1, 1.3, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0)
PAIRS = [(a, b) for i, a in enumerate(GRID) for b in GRID[i + 1:]]
IT2_KEYS = [(a, b, mode) for (a, b) in PAIRS for mode in ("m1", "mid", "m2")]
DEC_KEYS = [(a, b) for a in GRID for b in GRID]
SYN = ("equal", "volume", "noise", "density", "volume3")
N_SYN, N_REAL, N_INIT, TOL = 40, 20, 2, 1e-6

def instance(which, s, data_dir):
    if which in SYN:
        X, y, c, C = scenario(which, 1000 + s); return X, y, C
    X, y, C = real_dataset3(which, data_dir); r = np.random.default_rng(3000 + s)
    idx = np.sort(np.concatenate([r.choice(np.where(y == k)[0], int(round(0.75 * (y == k).sum())), replace=False) for k in range(C)]))
    return X[idx], y[idx], C

def run_instance(X, y, C, s):
    order = np.argsort(X, axis=0, kind="stable")
    rec = dict(fcm={m: 0.0 for m in GRID}, it2={k: 0.0 for k in IT2_KEYS}, dec={k: 0.0 for k in DEC_KEYS},
               mism_points=0, mism_runs=0, same_as_nearest_runs=0)
    for q in range(N_INIT):
        V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
        for m in GRID:
            rec["fcm"][m] += accuracy_h(y, run_fcm(X, V0, m, tol=TOL)[1], C) / N_INIT
        for k in DEC_KEYS:
            rec["dec"][k] += accuracy_h(y, run_fcm_decoupled(X, V0, k[0], k[1], tol=TOL)[1], C) / N_INIT
        for (a, b, mode) in IT2_KEYS:
            mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
            V, hp, hn = run_it2fcm_fast(X, V0, a, b, mc=mc, tol=TOL, order=order)
            rec["it2"][(a, b, mode)] += accuracy_h(y, hp, C) / N_INIT
            d = int((hp != hn).sum()); rec["mism_points"] += d; rec["mism_runs"] += int(d > 0)
    return rec

In [ ]:
from PIL import Image
def fmeasure_at(hist_ink, hist_bg, t):
    k = int(np.clip(np.ceil(t * 255.0), 0, 256))
    tp = hist_ink[:k].sum(); fp = hist_bg[:k].sum(); fn = hist_ink.sum() - tp
    return float(2 * tp / max(2 * tp + fp + fn, 1))

In [ ]:
X, y, c, C = scenario("volume", 1); V0 = init_pp(X, C, np.random.default_rng(0))
Vf, pf = run_fcm(X, V0, 2.0); Vi, hp, hn = run_it2fcm(X, V0, 2.0, 2.0, mc=2.0)
chk1 = float(np.abs(Vi - Vf).max()); print("m1 = m2 reproduces FCM:", chk1, bool((hp == pf).all()))
import itertools
r = np.random.default_rng(3); x = r.normal(size=9); wl = r.uniform(0, .5, 9); wu = wl + r.uniform(0, .5, 9)
vL, vR, _, _ = km_interval_1d(x, wl, wu)
vals = [np.sum(x * np.where(b, wu, wl)) / np.sum(np.where(b, wu, wl)) for b in itertools.product([0, 1], repeat=9)]
chk2 = max(abs(vL - min(vals)), abs(vR - max(vals))); print("KM equals brute force over 512 embedded sets:", chk2)
V1, h1, n1 = run_it2fcm(X, V0, 1.5, 5.0); V2, h2, n2 = run_it2fcm_fast(X, V0, 1.5, 5.0)
chk3 = float(np.abs(V1 - V2).max()); print("vectorized update equals reference:", chk3)
assert chk1 < 1e-12 and chk2 < 1e-12 and chk3 < 1e-9

In [ ]:
def run_dataset(nm):
    path = os.path.join(OUTDIR, f"runs05_{nm}.pkl")
    out = pickle.load(open(path, "rb")) if os.path.exists(path) else {}
    t0 = time.time()
    if nm == "dibco":
        P = sorted((f, f.replace("_source.", "_target.")) for f in glob.glob(os.path.join(DIBCO_DIR, "*DIBCO*_source.*")))
        P = [(a, b) for a, b in P if os.path.exists(b)]
        todo = [i for i in range(len(P)) if i not in out]
        for i in todo:
            f, g = P[i]
            I8 = np.array(Image.open(f).convert("L")); G = np.array(Image.open(g).convert("L")) < 128
            hist_ink = np.bincount(I8[G].ravel(), minlength=256).astype(float)
            hist_bg = np.bincount(I8[~G].ravel(), minlength=256).astype(float)
            r = np.random.default_rng(5000 + i)
            X = (I8.ravel()[r.choice(I8.size, min(2000, I8.size), replace=False)] / 255.0)[:, None]
            order = np.argsort(X, axis=0, kind="stable")
            F = lambda V: fmeasure_at(hist_ink, hist_bg, float(V.ravel().mean()))
            rec = dict(fcm={m: 0.0 for m in GRID}, it2={k: 0.0 for k in IT2_KEYS}, dec={k: 0.0 for k in DEC_KEYS})
            for q in range(N_INIT):
                V0 = init_pp(X, 2, np.random.default_rng(10_000 * (q + 1) + i))
                for m in GRID: rec["fcm"][m] += F(run_fcm(X, V0, m, tol=TOL)[0]) / N_INIT
                for k in DEC_KEYS: rec["dec"][k] += F(run_fcm_decoupled(X, V0, k[0], k[1], tol=TOL)[0]) / N_INIT
                for (a, b, mode) in IT2_KEYS:
                    mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
                    rec["it2"][(a, b, mode)] += F(run_it2fcm_fast(X, V0, a, b, mc=mc, tol=TOL, order=order)[0]) / N_INIT
            out[i] = dict(rec=rec, C=2, n=int(I8.size), name=os.path.basename(f).replace("TRAIN_", "").replace("_source", ""))
            pickle.dump(out, open(path, "wb"))
    else:
        n = (N_SYN if nm in SYN else N_REAL)
        todo = [s for s in range(n) if s not in out]
        for s in todo:
            X, y, C = instance(nm, s, DATA_DIR)
            out[s] = dict(rec=run_instance(X, y, C, s), C=C, n=len(X))
            pickle.dump(out, open(path, "wb"))
    print(f"{nm:10s} {len(out)} instances ({len(todo)} new) in {time.time()-t0:.0f}s", flush=True)

for nm in DATASETS:
    run_dataset(nm)

In [ ]:
import numpy as np, pickle, json, os
from scipy import stats
SYN = ("equal", "volume", "noise", "density", "volume3")
REAL = ("iris", "wine", "wdbc", "wbc", "sonar", "pima", "seeds", "glass", "ionosphere", "banknote", "ecoli", "dibco")
NAMES = SYN + REAL
def tp(d):
    d = np.asarray(d, float)
    if np.allclose(d, 0): return dict(mean=0.0, t_p=1.0, w_p=1.0, nz=0, n=len(d))
    t = float(stats.ttest_1samp(d, 0).pvalue)
    try: w = float(stats.wilcoxon(d).pvalue)
    except ValueError: w = 1.0
    return dict(mean=float(d.mean()), t_p=t, w_p=w if np.isfinite(w) else 1.0, nz=int((np.abs(d) > 1e-12).sum()), n=len(d))
def verdict(a, b):
    sig = lambda z: z["t_p"] < 0.05 and z["w_p"] < 0.05
    if sig(a) and sig(b) and a["mean"] > 0 and b["mean"] > 0: return "BETTER"
    if sig(a) and sig(b) and a["mean"] < 0 and b["mean"] < 0: return "WORSE"
    return "n.s."
out = {}
for nm in NAMES:
    f = os.path.join(OUTDIR, f"runs05_{nm}.pkl")
    if not os.path.exists(f): continue
    R = pickle.load(open(f, "rb")); ids = sorted(R); h = len(ids) // 2; A, B = ids[:h], ids[h:]
    if nm == "dibco":
        rng = np.random.default_rng(7); years = {i: R[i]["name"][5:9] for i in ids}; fold = {}
        for yv in sorted(set(years.values())):
            ix = [i for i in ids if years[i] == yv]; ix = [ix[j] for j in rng.permutation(len(ix))]
            for j, i in enumerate(ix): fold[i] = j % 2
        A = [i for i in ids if fold[i] == 0]; B = [i for i in ids if fold[i] == 1]
    fam = {"FCM": ("fcm", list(GRID)), "IT2FCM": ("it2", IT2_KEYS), "T1dec": ("dec", DEC_KEYS)}
    o = dict(n_sets=len(ids), C=R[ids[0]]["C"], splits={})
    for sn, (tr, te) in (("A->B", (A, B)), ("B->A", (B, A))):
        sel = {f: max(keys, key=lambda k: np.mean([R[i]["rec"][key][k] for i in tr])) for f, (key, keys) in fam.items()}
        test = {f: np.array([R[i]["rec"][fam[f][0]][sel[f]] for i in te]) for f in fam}
        o["splits"][sn] = dict(selected={f: str(sel[f]) for f in sel}, acc={f: float(test[f].mean()) for f in fam},
                               it2_vs_fcm=tp(test["IT2FCM"] - test["FCM"]), it2_vs_dec=tp(test["IT2FCM"] - test["T1dec"]),
                               dec_vs_fcm=tp(test["T1dec"] - test["FCM"]))
    a, b = o["splits"]["A->B"], o["splits"]["B->A"]
    o["verdict"] = {k: verdict(a[k], b[k]) for k in ("it2_vs_fcm", "it2_vs_dec", "dec_vs_fcm")}
    o["hr_design"] = tp([max(R[i]["rec"]["it2"][k] for k in IT2_KEYS if k[2] == "mid") - R[i]["rec"]["fcm"][2.0] for i in ids])
    o["oracle_it2_vs_dec"] = tp([max(R[i]["rec"]["it2"].values()) - max(R[i]["rec"]["dec"].values()) for i in ids])
    o["partition_vs_nearest"] = dict(runs_with_mismatch=int(sum(R[i]["rec"].get("mism_runs", 0) for i in ids)),
                                     points=int(sum(R[i]["rec"].get("mism_points", 0) for i in ids)),
                                     runs=len(ids) * len(IT2_KEYS) * 2)
    edge = lambda s: any(str(e) in s for e in ("1.05", "20.0"))
    o["selected_on_grid_edge"] = {f: [edge(o["splits"][sp]["selected"][f]) for sp in o["splits"]] for f in fam}
    out[nm] = o
print(f"{'data':8s} {'IT2 vs FCM':>28s} {'IT2 vs T1dec':>28s} {'T1dec vs FCM':>28s} {'Hwang-Rhee design':>18s}  partition!=nearest")
for nm, o in out.items():
    a, b = o["splits"]["A->B"], o["splits"]["B->A"]
    cell = lambda k: f"{o['verdict'][k]:6s} {a[k]['mean']*100:+.2f}/{b[k]['mean']*100:+.2f}"
    hr = o["hr_design"]
    pv = o["partition_vs_nearest"]
    print(f"{nm:8s} {cell('it2_vs_fcm'):>28s} {cell('it2_vs_dec'):>28s} {cell('dec_vs_fcm'):>28s} {hr['mean']*100:+7.2f} (p {hr['t_p']:.0e})  {pv['points']} pts / {pv['runs_with_mismatch']} of {pv['runs']} runs")
print()
for nm, o in out.items():
    print(f"{nm:8s} selected: " + " | ".join(f"{sp}: " + ", ".join(f"{f}={v}" for f, v in o['splits'][sp]['selected'].items()) for sp in o['splits']))

print()
for label, group in (("all", NAMES), ("real only", REAL)):
    rows = [n for n in group if n in out]
    for key in ("it2_vs_fcm", "it2_vs_dec"):
        d = np.array([np.mean([out[n]["splits"][sp][key]["mean"] for sp in ("A->B", "B->A")]) for n in rows])
        w = stats.wilcoxon(d).pvalue if np.any(np.abs(d) > 1e-12) else 1.0
        print(f"data-set level [{label}, {len(rows)} sets] {key}: median {np.median(d)*100:+.2f} pp, positive {int((d>0).sum())}, negative {int((d<0).sum())}, Wilcoxon p = {w:.3g}")
        out.setdefault("_dataset_level", {})[f"{label}:{key}"] = dict(n=len(rows), median=float(np.median(d)), pos=int((d>0).sum()), neg=int((d<0).sum()), wilcoxon_p=float(w))

RESULTS = out

In [ ]:
import ast
DECOMP = {}
for nm in ("ecoli", "glass"):
    if nm not in RESULTS: continue
    R = pickle.load(open(os.path.join(OUTDIR, f"runs05_{nm}.pkl"), "rb")); ids = sorted(R); h = len(ids) // 2
    for sn, te in (("A->B", ids[h:]), ("B->A", ids[:h])):
        a, b, mode = ast.literal_eval(RESULTS[nm]["splits"][sn]["selected"]["IT2FCM"]); mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
        mf = float(RESULTS[nm]["splits"][sn]["selected"]["FCM"])
        hard, near, fcm_ = [], [], []
        for s in te:
            X, y, C = instance(nm, s, DATA_DIR); order = np.argsort(X, 0, kind="stable"); ap = an_ = 0.0
            for q in range(N_INIT):
                V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
                V, hp, hn = run_it2fcm_fast(X, V0, a, b, mc=mc, tol=TOL, order=order)
                ap += accuracy_h(y, hp, C) / N_INIT; an_ += accuracy_h(y, hn, C) / N_INIT
            hard.append(ap); near.append(an_); fcm_.append(R[s]["rec"]["fcm"][mf])
        DECOMP[f"{nm} {sn}"] = dict(hard=float(np.mean(hard)), nearest=float(np.mean(near)), fcm=float(np.mean(fcm_)))
        print(f"{nm:6s} {sn}: IT2 hard partition {np.mean(hard):.4f} | IT2 nearest prototype {np.mean(near):.4f} | FCM tuned {np.mean(fcm_):.4f}")

In [ ]:
LINES = []
def say(s=""): print(s); LINES.append(s)
names = [n for n in NAMES if n in RESULTS]
say("=" * 78); say("05 -- main runs, alternative reading of the bounds"); say(f"run {RUN_STAMP}"); say("=" * 78)
say(f"data sets: {len(names)} ({', '.join(names)})"); say("")
for key, label in (("it2_vs_fcm", "IT2FCM vs FCM"), ("it2_vs_dec", "IT2FCM vs type-1 decoupled"), ("dec_vs_fcm", "type-1 decoupled vs FCM")):
    v = [RESULTS[n]["verdict"][key] for n in names]
    say(f"{label:28s}: better {v.count('BETTER'):2d}  worse {v.count('WORSE'):2d}  n.s. {v.count('n.s.'):2d}   "
        f"[better: {', '.join(n for n in names if RESULTS[n]['verdict'][key]=='BETTER') or '-'}; "
        f"worse: {', '.join(n for n in names if RESULTS[n]['verdict'][key]=='WORSE') or '-'}]")
say("")
for k, v in RESULTS.get("_dataset_level", {}).items():
    say(f"data-set level {k:24s}: median {v['median']*100:+.2f} pp, +{v['pos']} / -{v['neg']}, Wilcoxon p = {v['wilcoxon_p']:.3g}")
say("")
hr = [RESULTS[n]["hr_design"] for n in names]
say(f"comparison design of Hwang and Rhee (best IT2 vs FCM at m=2): significant in {sum(h['t_p'] < 0.05 for h in hr)}/{len(hr)}, "
    f"gain {min(h['mean'] for h in hr)*100:+.2f} to {max(h['mean'] for h in hr)*100:+.2f} pp")
two = [n for n in names if RESULTS[n]["C"] == 2 and n != "dibco"]
say(f"hard partition vs nearest prototype, C = 2: {sum(RESULTS[n]['partition_vs_nearest']['points'] for n in two)} differing points "
    f"over {sum(RESULTS[n]['partition_vs_nearest']['runs'] for n in two)} runs")
for n in names:
    if RESULTS[n]["C"] > 2:
        pv = RESULTS[n]["partition_vs_nearest"]
        say(f"    C = {RESULTS[n]['C']} {n:8s}: {pv['points']} points in {pv['runs_with_mismatch']} of {pv['runs']} runs")
for k, v in DECOMP.items():
    say(f"decomposition {k}: hard partition {v['hard']:.4f}, nearest prototype {v['nearest']:.4f}, FCM {v['fcm']:.4f}")
payload = dict(notebook="05", run_stamp=RUN_STAMP,
               results=RESULTS, decomposition=DECOMP)
report = "\n".join(LINES) + "\n"
paths = [os.path.join(OUTDIR, "results_05.json"), os.path.join(OUTDIR, "results_05_report.txt"),
         os.path.join(OUTDIR, "reports", f"results_05_{RUN_STAMP}.json"),
         os.path.join(OUTDIR, "reports", f"results_05_report_{RUN_STAMP}.txt")]
for p in paths:
    with open(p, "w") as fh:
        if p.endswith(".json"): json.dump(payload, fh, indent=1, default=str)
        else: fh.write(report)
        fh.flush(); os.fsync(fh.fileno())
print()
for p in paths:
    ok = os.path.exists(p) and os.path.getsize(p) > 0
    print(f"{'written' if ok else 'NOT WRITTEN'}: {p} ({os.path.getsize(p) if ok else 0} bytes)")